# Notebook 2 — Preprocessing, Smart Augmentation & DataLoaders

**Prerequisites:** NB1 must have run and saved `train_data.npz`, `val_data.npz`, `test_data.npz`, `metadata.json`

## Changes from Previous Version
1. ✅ **Metadata tensors** (age, sex, heart rate) are now part of every DataLoader batch
2. ✅ Augmentation-based oversampling (NOT SMOTE on 12000-dim vectors)
3. ✅ WeightedRandomSampler for balanced batches every epoch
4. ✅ NaN-safe per-sample Z-score normalization
5. ✅ `drop_last=True` in all DataLoaders — no BatchNorm crash
6. ✅ Transpose applied in `__getitem__`: stored (1000,12) → returned (12,1000) for Conv1d model
7. ✅ MixUp implemented as a standalone function for use in NB4 training loop


In [ ]:
from pathlib import Path
import os, gc

SAVE_DIR = Path("./ECG_Project/data")

gc.collect()

# Clean up only the preprocessing outputs (not NB1 outputs)
files_to_delete = [
    "train_processed.npz",
    "val_processed.npz",
    "test_processed.npz",
    "class_weights.pt",
    "loader_config.json",
]
for f in files_to_delete:
    p = SAVE_DIR / f
    if p.exists():
        try:
            os.remove(p)
            print(f"🗑️  Deleted: {f}")
        except PermissionError:
            print(f"❌ Locked: {f} — close any program using it and retry")
    else:
        print(f"⏭️  Not found (skip): {f}")

print("\n✅ Clean. Proceeding with NB2.")


In [ ]:
import numpy as np
import json, warnings
from pathlib import Path
from collections import Counter
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = Path("./ECG_Project/data")

with open(SAVE_DIR / "metadata.json") as f:
    META = json.load(f)
RISK_LABELS = {int(k): v for k, v in META["risk_labels"].items()}
TARGET_LEN  = META["target_len"]
NUM_LEADS   = META["num_leads"]

print(f"Device: {device}")
print(f"Input shape (model): ({NUM_LEADS}, {TARGET_LEN})")


## Step 1: Load Raw Splits

In [ ]:
data = {}
for split in ["train", "val", "test"]:
    d = np.load(SAVE_DIR / f"{split}_data.npz")
    # X: (N, 1000, 12)   meta: (N, 3)  [age_norm, sex, hr_norm]
    data[split] = (
        d["X"].astype(np.float32),
        d["y"].astype(np.int64),
        d["meta"].astype(np.float32) if "meta" in d else np.zeros((len(d["y"]), 3), np.float32)
    )

X_train, y_train, M_train = data["train"]
X_val,   y_val,   M_val   = data["val"]
X_test,  y_test,  M_test  = data["test"]

print("Original class distribution in TRAINING set:")
cnt = Counter(y_train)
for k in range(4):
    n = cnt[k]
    print(f"  {RISK_LABELS[k]:10s}: {n:5,}  ({100*n/len(y_train):.1f}%)")


## Step 2: NaN-Safe Per-Sample Normalization

Applied again here as a safety pass (NB1 already normalised, but augmentation may reintroduce edge cases).


In [ ]:
def normalize_ecg(X):
    """Per-sample Z-score normalization, NaN-safe. X: (N, T, L)"""
    mean = X.mean(axis=1, keepdims=True)              # (N,1,L)
    std  = X.std(axis=1,  keepdims=True)              # (N,1,L)
    std  = np.where(std < 1e-8, 1e-8, std)
    X    = (X - mean) / std
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

X_train = normalize_ecg(X_train)
X_val   = normalize_ecg(X_val)
X_test  = normalize_ecg(X_test)

print(f"NaN in train: {np.isnan(X_train).sum()}  ✅")
print(f"NaN in val  : {np.isnan(X_val).sum()}  ✅")
print(f"NaN in test : {np.isnan(X_test).sum()}  ✅")
print("✅ Normalization complete.")


## Step 3: Augmentation-Based Oversampling (minority classes only)

Upsample Moderate and High classes using noise injection, amplitude scaling, and time shift.  
Never applied to val/test sets.


In [ ]:
def augment_ecg(x):
    """Apply random augmentations. x: (1000, 12) numpy array."""
    x = x.copy()
    if np.random.rand() < 0.7:
        x += np.random.normal(0, 0.01, x.shape).astype(np.float32)
    if np.random.rand() < 0.7:
        x *= np.random.uniform(0.85, 1.15)
    if np.random.rand() < 0.5:
        x = np.roll(x, np.random.randint(-30, 30), axis=0)
    if np.random.rand() < 0.15:
        x[:, np.random.randint(0, 12)] = 0.0  # random lead dropout
    # Renormalise after augmentation
    mean = x.mean(axis=0, keepdims=True)
    std  = np.where(x.std(axis=0, keepdims=True) < 1e-8, 1e-8,
                    x.std(axis=0, keepdims=True))
    x = (x - mean) / std
    return np.nan_to_num(x, nan=0.0).astype(np.float32)


def oversample_classes(X, y, M, target_ratio=0.5):
    """
    Augment minority classes until each class has at least
    target_ratio * majority_class_count samples.
    """
    cnt     = Counter(y)
    max_cnt = max(cnt.values())
    X_extra, y_extra, M_extra = [], [], []

    for cls in range(4):
        n_current = cnt.get(cls, 0)
        n_target  = int(max_cnt * target_ratio)
        if n_current >= n_target:
            continue
        n_needed  = n_target - n_current
        cls_idx   = np.where(y == cls)[0]
        chosen    = np.random.choice(cls_idx, n_needed, replace=True)
        for i in chosen:
            X_extra.append(augment_ecg(X[i]))
            y_extra.append(cls)
            M_extra.append(M[i])

    if X_extra:
        X = np.vstack([X, np.stack(X_extra)])
        y = np.concatenate([y, np.array(y_extra, dtype=np.int64)])
        M = np.vstack([M, np.stack(M_extra)])

    # Shuffle
    perm = np.random.permutation(len(y))
    return X[perm], y[perm], M[perm]


X_train_aug, y_train_aug, M_train_aug = oversample_classes(X_train, y_train, M_train)

print("After oversampling:")
cnt = Counter(y_train_aug)
for k in range(4):
    n = cnt.get(k, 0)
    print(f"  {RISK_LABELS[k]:10s}: {n:5,}")


## Step 4: Save Processed Splits

In [ ]:
# Save in (N, 1000, 12) format; the Dataset class transposes to (12, 1000) in __getitem__
np.savez_compressed(SAVE_DIR / "train_processed.npz", X=X_train_aug, y=y_train_aug, meta=M_train_aug)
np.savez_compressed(SAVE_DIR / "val_processed.npz",   X=X_val,       y=y_val,       meta=M_val)
np.savez_compressed(SAVE_DIR / "test_processed.npz",  X=X_test,      y=y_test,      meta=M_test)
print("✅ Processed splits saved.")


## Step 5: Class Weights for CB-Focal Loss

In [ ]:
classes     = np.arange(4)
cw          = compute_class_weight("balanced", classes=classes, y=y_train_aug)
class_weights = torch.tensor(cw, dtype=torch.float32)
torch.save(class_weights, SAVE_DIR / "class_weights.pt")

print("Class weights (inverse frequency):")
for k, w in enumerate(class_weights):
    print(f"  {RISK_LABELS[k]:10s}: {w:.4f}")


## Step 6: ECGDataset Class (with Metadata)

Each batch returns `(ecg_tensor, meta_tensor, label_tensor)`.  
The metadata tensor feeds the Metadata MLP fusion branch in Tier 3.


In [ ]:
class ECGDataset(Dataset):
    """
    ECG Dataset with metadata support.
    Stored as (N, 1000, 12), returned as (12, 1000) for Conv1d model.
    __getitem__ returns: (ecg (12,1000), meta (3,), label (int))
    """
    def __init__(self, X, y, meta, augment=False):
        self.X       = X.astype(np.float32)     # (N, 1000, 12)
        self.y       = y.astype(np.int64)
        self.meta    = meta.astype(np.float32)  # (N, 3)
        self.augment = augment

    def _augment(self, x):
        """Light online augmentation during training."""
        x = x.copy()
        if np.random.rand() < 0.5:
            x += np.random.normal(0, 0.01, x.shape).astype(np.float32)
        if np.random.rand() < 0.5:
            x *= np.random.uniform(0.9, 1.1)
        if np.random.rand() < 0.5:
            x = np.roll(x, np.random.randint(-20, 20), axis=0)
        if np.random.rand() < 0.1:
            x[:, np.random.randint(0, 12)] = 0.0
        mean = x.mean(axis=0, keepdims=True)
        std  = np.where(x.std(axis=0, keepdims=True) < 1e-8, 1e-8,
                        x.std(axis=0, keepdims=True))
        x = (x - mean) / std
        return np.nan_to_num(x, nan=0.0).astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].copy()   # (1000, 12)
        if self.augment:
            x = self._augment(x)
        # Transpose: (1000, 12) → (12, 1000) — correct shape for Conv1d model
        ecg_t  = torch.from_numpy(x.T)              # (12, 1000)
        meta_t = torch.from_numpy(self.meta[idx])   # (3,)
        label  = torch.tensor(self.y[idx], dtype=torch.long)
        return ecg_t, meta_t, label


print("✅ ECGDataset defined.")
# Quick shape check
dummy_ds = ECGDataset(X_train_aug[:4], y_train_aug[:4], M_train_aug[:4])
ecg, meta, label = dummy_ds[0]
print(f"   ECG shape  : {ecg.shape}   ← should be (12, 1000)")
print(f"   Meta shape : {meta.shape}  ← should be (3,)")
print(f"   Label      : {label}")


## Step 7: MixUp Utility

MixUp is applied at the batch level in the NB4 training loop (not here).  
Defined here and referenced from NB4.


In [ ]:
def mixup_batch(ecg, meta, labels, alpha=0.2):
    """
    Apply MixUp augmentation to a batch.
    ecg:    (B, 12, 1000)
    meta:   (B, 3)
    labels: (B,) long tensor
    Returns: mixed_ecg, mixed_meta, labels_a, labels_b, lam
    """
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    B   = ecg.size(0)
    idx = torch.randperm(B, device=ecg.device)
    mixed_ecg  = lam * ecg  + (1 - lam) * ecg[idx]
    mixed_meta = lam * meta + (1 - lam) * meta[idx]
    return mixed_ecg, mixed_meta, labels, labels[idx], lam


print("✅ MixUp utility defined.")


## Step 8: Build DataLoaders

In [ ]:
BATCH_SIZE = 64

# WeightedRandomSampler for balanced batches in training
sample_weights = torch.zeros(len(y_train_aug))
for cls in range(4):
    n = (y_train_aug == cls).sum()
    sample_weights[y_train_aug == cls] = 1.0 / max(n, 1)
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_ds = ECGDataset(X_train_aug, y_train_aug, M_train_aug, augment=True)
val_ds   = ECGDataset(X_val,       y_val,       M_val,       augment=False)
test_ds  = ECGDataset(X_test,      y_test,      M_test,      augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=0, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True, drop_last=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True, drop_last=False)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")

# Verify batch shape
ecg_b, meta_b, y_b = next(iter(train_loader))
print(f"\nBatch ECG  shape : {ecg_b.shape}    ← should be (64, 12, 1000)")
print(f"Batch Meta shape : {meta_b.shape}   ← should be (64, 3)")
print(f"Batch Label shape: {y_b.shape}")


## Step 9: Save Loader Config

In [ ]:
loader_config = {
    "batch_size"       : BATCH_SIZE,
    "num_workers"      : 0,
    "drop_last_train"  : True,
    "mixup_alpha"      : 0.2,
    "oversample_ratio" : 0.5,
    "augment_train"    : True,
    "meta_dim"         : 3,
    "meta_features"    : ["age_norm", "sex", "hr_norm"],
}
with open(SAVE_DIR / "loader_config.json", "w") as f:
    json.dump(loader_config, f, indent=2)

print("NOTEBOOK 2 COMPLETE ✅")
print("Outputs:")
print(f"  {SAVE_DIR}/train_processed.npz")
print(f"  {SAVE_DIR}/val_processed.npz")
print(f"  {SAVE_DIR}/test_processed.npz")
print(f"  {SAVE_DIR}/class_weights.pt")
print(f"  {SAVE_DIR}/loader_config.json")
print()
print("Next → Notebook 3: New ECGRiskNet-XAI Model Architecture")
